<a href="https://colab.research.google.com/github/Om-merkle/AI_EMBEDDING_FINETUNE/blob/main/MED_EMBED_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Medical Text Embedding Project
This notebook demonstrates how to use the [MedEmbed-small-v0.1](https://huggingface.co/abhinand/MedEmbed-small-v0.1) model for generating medical text embeddings.
```

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install datasets

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

# Load the specific medical embedding model
model_name = 'abhinand/MedEmbed-small-v0.1'
model = SentenceTransformer(model_name)

print(f"Model '{model_name}' loaded successfully.")

```markdown
### Prepare Dataset
We will create a sample medical dataset to demonstrate the embedding process.
```

In [ ]:
# Creating a sample medical dataset
data = {
    'text': [
        'What are the symptoms of Type 2 Diabetes?',
        'Treatment options for hypertension.',
        'Common side effects of ibuprofen.',
        'How to manage asthma in children?',
        'The role of insulin in blood sugar regulation.'
    ],
    'category': ['Diabetes', 'Hypertension', 'Pharmacology', 'Respiratory', 'Diabetes']
}

df = pd.DataFrame(data)
display(df)

```markdown
### Generate Embeddings
Now we encode the text into high-dimensional vectors.
```

In [ ]:
# Generate embeddings
embeddings = model.encode(df['text'].tolist())

# Add embeddings to the dataframe (as a list for visualization)
df['embedding'] = list(embeddings)

print(f"Embedding shape: {embeddings.shape}")
display(df.head())

```markdown
## Domain-Specific MTEB-Style Benchmarking Framework
This framework allows you to shortlist models by evaluating their performance on domain-specific triplets (Query, Positive, Negative).
```

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
import pandas as pd

class EmbeddingBenchmarker:
    def __init__(self, model_names, device='cpu'):
        print("Initializing models...")
        self.models = {name: SentenceTransformer(name, device=device) for name in model_names}

    def evaluate_triplets(self, triplets_df):
        """
        Evaluates models on (query, positive, negative) triplets.
        Metric: % of cases where similarity(Q, P) > similarity(Q, N)
        """
        results = {}

        queries = triplets_df['query'].tolist()
        positives = triplets_df['positive'].tolist()
        negatives = triplets_df['negative'].tolist()

        for name, model in self.models.items():
            print(f"Evaluating {name}...")
            q_emb = model.encode(queries, convert_to_tensor=True)
            p_emb = model.encode(positives, convert_to_tensor=True)
            n_emb = model.encode(negatives, convert_to_tensor=True)

            # Calculate cosine similarities
            pos_sim = util.cos_sim(q_emb, p_emb).diagonal()
            neg_sim = util.cos_sim(q_emb, n_emb).diagonal()

            # Accuracy = how often positive is closer than negative
            accuracy = (pos_sim > neg_sim).float().mean().item()
            results[name] = accuracy

        return results

# Define models to compare: General Baseline vs Domain-Specific
models_to_test = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "abhinand/MedEmbed-small-v0.1"
]

benchmarker = EmbeddingBenchmarker(models_to_test)

```markdown
### Define/Load Triplet Dataset
Once you provide the link, we can use `datasets.load_dataset()` here. For now, we use a placeholder triplet set.
```

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# Scaling up the sample size to 200 for a more significant benchmark
try:
    print("Downloading and loading a larger sample (200) from Hugging Face...")
    dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split='train[:200]')

    raw_df = dataset.to_pandas()

    # Mapping the dataset to the triplet format
    # Using a seeded random sample for the negative to ensure reproducibility
    triplets_df = pd.DataFrame({
        'query': raw_df['instruction'],
        'positive': raw_df['output'],
        'negative': raw_df['output'].sample(frac=1, random_state=42).reset_index(drop=True)
    })

    print(f"Successfully loaded and formatted {len(triplets_df)} medical triplets.")
except Exception as e:
    print(f"Failed to load dataset: {e}")

In [ ]:
# Placeholder Triplet Dataset
data_triplets = {
    "query": [
        "What is the main cause of type 2 diabetes?",
        "How to treat acute hypertension?"
    ],
    "positive": [
        "Type 2 diabetes is primarily caused by insulin resistance and genetics.",
        "Acute hypertension is managed with rapid-acting antihypertensive agents."
    ],
    "negative": [
        "The weather in London is often rainy during the spring.",
        "Baking a cake requires flour, sugar, and eggs."
    ]
}

triplets_df = pd.DataFrame(data_triplets)

# Run Evaluation
scores = benchmarker.evaluate_triplets(triplets_df)

# Display Shortlist Results
report_df = pd.DataFrame(list(scores.items()), columns=['Model', 'Triplet Accuracy'])
display(report_df.sort_values(by='Triplet Accuracy', ascending=False))

```markdown
## 🏆 Domain Embedding Leaderboard
Below is the formatted leaderboard comparing the models based on the domain-specific triplet evaluation.
```

In [ ]:
import numpy as np

# Re-running the leaderboard generation with the expanded dataset
leaderboard_output = generate_leaderboard(benchmarker, triplets_df)
leaderboard_output

In [ ]:
import numpy as np
from sentence_transformers import util

def calculate_average_margins(benchmarker, triplets_df):
    results = []
    queries = triplets_df['query'].tolist()
    positives = triplets_df['positive'].tolist()
    negatives = triplets_df['negative'].tolist()

    print("Calculating margins for all models...")

    for name, model in benchmarker.models.items():
        # Encode all components
        q_emb = model.encode(queries, convert_to_tensor=True)
        p_emb = model.encode(positives, convert_to_tensor=True)
        n_emb = model.encode(negatives, convert_to_tensor=True)

        # Calculate pairwise cosine similarities (diagonal of the matrix)
        pos_sims = util.cos_sim(q_emb, p_emb).diagonal()
        neg_sims = util.cos_sim(q_emb, n_emb).diagonal()

        # Compute individual margins and then the mean
        margins = (pos_sims - neg_sims).cpu().numpy()
        avg_margin = np.mean(margins)

        results.append({
            "Model": name,
            "Average Margin": avg_margin,
            "Max Margin": np.max(margins),
            "Min Margin": np.min(margins)
        })

    return pd.DataFrame(results)

# Calculate and display
margin_report = calculate_average_margins(benchmarker, triplets_df)
display(margin_report.sort_values(by='Average Margin', ascending=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_margin_distribution(benchmarker, triplets_df):
    all_margins = []
    queries = triplets_df['query'].tolist()
    positives = triplets_df['positive'].tolist()
    negatives = triplets_df['negative'].tolist()

    for name, model in benchmarker.models.items():
        # Get embeddings
        q_emb = model.encode(queries, convert_to_tensor=True)
        p_emb = model.encode(positives, convert_to_tensor=True)
        n_emb = model.encode(negatives, convert_to_tensor=True)

        # Calculate cosine similarities
        pos_sims = util.cos_sim(q_emb, p_emb).diagonal()
        neg_sims = util.cos_sim(q_emb, n_emb).diagonal()

        # Calculate margins
        margins = (pos_sims - neg_sims).cpu().numpy()

        # Store in a format suitable for long-form plotting
        for m in margins:
            all_margins.append({'Model': name.split('/')[-1], 'Margin': m})

    plot_df = pd.DataFrame(all_margins)

    # Plotting
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Model', y='Margin', data=plot_df, palette='Set2', hue='Model', legend=False)
    plt.axhline(0, color='red', linestyle='--', alpha=0.6, label='Zero Margin Boundary')
    plt.title('Distribution of Retrieval Margins (Pos_Sim - Neg_Sim)')
    plt.ylabel('Margin (Higher is better)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# Execute the visualization
plot_margin_distribution(benchmarker, triplets_df)